# Trade the 8-K — starter notebook

**Gator Quant Hacks 2026 · Massive Challenge**

Every material corporate event must be disclosed on a Form 8-K within four business days: a CEO
leaves, a merger is signed, guidance is cut, a new CFO is appointed. Massive's Filings & Disclosures
dataset tags every 8-K since January 2022 with one of 119 event types. The night before a filing hits
EDGAR, the options market has already put a price on how much the stock *should* move.

## The challenge

> **Pick one or more 8-K disclosure categories and one strategy from the library, and make the
> case that the category is a signal for entering that strategy.** Define the rule: which filings,
> which contracts, entered when, exited when. Test it on the 100 largest US companies across the
> fixed horizons, compare it with ordinary days for the same names, show it out of sample, and show
> how sensitive it is to your parameters.

You are judged on the novelty of the category-to-strategy link, the rigor of the analysis, whether
it replicates on a sealed window, the realism of the trade, and clarity. **Not on P&L.** A
well-argued null result with a clear decay curve beats a lucky backtest.

| Tier | What you deliver | Where this notebook gets you |
|---|---|---|
| **Baseline** (every team) | One category, one strategy, one expiry bucket, a written interpretation | Runs end to end as-is: CFO appointments × all five strategies × 3–6 month options, in-sample and out-of-sample |
| **Stretch** (separates the top teams) | Decay across horizons and expiry buckets · combined categories · a placebo control · timing refinements · a realistic trade specification | Sections 7, 10, 11 and the timing stretch |
| **Sealed window** (judges) | Judges rerun your pipeline on dates you never saw | Final section, one function call |

## How to use this notebook

1. **Run it once unchanged.** The worked example (`EVENT_TAG = "cfo_appointment"`) runs end to
   end in about 10 minutes and confirms your setup. Its result is a null: no strategy beats ordinary
   days with confidence. Reproducing it is not a submission.
2. **Form a hypothesis.** One or more of the 119 tags, one of the five strategies, and an economic
   reason the market should misprice that event.
3. **Change `EVENT_TAG`** in section 2 and rerun. To combine tags, build your own event table in
   section 4. Every table the rubric asks for is produced below: scoreboard with intervals (7),
   placebo gap (7), out-of-sample (8), sensitivity (10), trade costs (11).
4. **Check the neighbours.** A result that holds at one bucket, one OTM level or one horizon is not
   a finding; section 10 is where you show that.
5. **Write it up** (see *What you submit*) and hand in the notebook with the write-up.

The worked example iterates every CFO appointment at the 100 largest US companies in the in-sample
window, prices all five strategies on each from the option chain as it stood the session before the
filing, marks them at fixed horizons, compares them with the same strategies on ordinary days for the
same names, ranks them, and only then picks one ticker to walk through. Swap the tag, the strategy
set or the universe and the same pipeline runs. The windows are set in section 2.

## Data

Two datasets, one API key, all from `api.massive.com`:

| Dataset | Endpoint | Used for |
|---|---|---|
| Filings & Disclosures | [`/stocks/filings/8-K/vX/disclosures`](https://massive.com/docs/rest/stocks/filings/8-k-disclosures) · [taxonomy](https://massive.com/docs/rest/stocks/filings/disclosure-categories) | The events: 119 tagged 8-K event types, one row per event, coverage from January 2022 |
| Options contracts | [`/v3/reference/options/contracts`](https://massive.com/docs/rest/options/contracts/all-contracts) with `as_of` | The chain as it existed on the pre-event session |
| Options aggregates | [`/v2/aggs/ticker/O:…/range/1/day/…`](https://massive.com/docs/rest/options/aggregates/custom-bars) | Every leg's daily close and volume from entry to expiry, **and the stock price**, recovered from the chain by put-call parity |

There is no stock feed, index membership feed or earnings calendar in this challenge. The trading
calendar is built from NYSE holiday rules (section 3), the stock price is read off the option chain
(section 5), and the 100 shares behind three of the five strategies are replicated with options
(section 6).

Every API response is cached on disk in `.massive_cache/`, so reruns are fast and a sealed-window
rerun only fetches new dates.

## Vocabulary

| Term | What it means here |
|---|---|
| **Call** | The right to *buy* 100 shares at the strike, any time before expiry |
| **Put** | The right to *sell* 100 shares at the strike, any time before expiry |
| **Strike** | The fixed price in the contract. `O:AAPL260116C00150000` is a $150 call expiring 2026-01-16 |
| **Expiry / DTE** | When the contract dies. `EXPIRY_BUCKETS` picks contracts by days-to-expiry from the pre-event session |
| **Premium** | What the contract costs. Quoted per share — **multiply by 100** for one contract |
| **At the money (ATM)** | Strike closest to the current stock price |
| **Debit / credit** | You pay to open (debit) or you are paid to open (credit) |
| **Long / short a leg** | You bought it (long, a right) or you sold it (short, an *obligation* — you keep the premium either way) |
| **Covered / secured** | A short leg backed by what it obliges you to deliver: shares for a short call, cash for a short put. Four of the five strategies are covered or secured |
| **Break-even** | Where the stock must be at expiry for the trade to return zero |
| **Implied move** | What the chain says the event is worth: the cost of the ATM call plus the ATM put, divided by spot. The benchmark every result here is measured against; section 7 reports it |
| **IV** | Implied volatility. It usually *falls* once the news is out — see "the volatility trap" below |
| **Assignment** | A short leg exercised against you. US equity options are American-style, so it can happen early — and a short call is most at risk the day before a dividend goes ex |

**The stock leg.** Covered call, protective put and collar start with "own 100 shares", and there
is no stock data in this challenge. A long call and a short put at the same strike and expiry is a
*synthetic long*: by put-call parity it moves one for one with the stock. Section 6 builds it from
the at-the-money pair and uses it wherever a strategy needs shares. The appendix lists what the
substitution leaves out (dividends and early assignment).

**Implied vs realized.** Every strategy is a bet on the implied move (what the chain charged on the
pre-event session) against the realized move (what the stock did). Buying options wins when realized
> implied; selling wins when realized < implied. Section 7 computes the ratio. The long call is the
only one of the five that wants a big move; the other four do better when the event is less dramatic
than the chain implied. A thesis of "the market over-pays for this category" has four strategies
built to harvest it, and is a less crowded angle than a long-volatility event study.

**The volatility trap.** "The stock moved a lot and my long call still lost money" is common and
not a contradiction: implied volatility collapses once the uncertainty resolves, and the option gives
back in IV what it gained in direction. The long call is exposed to this; the covered call and
cash-secured put are paid by it; the protective put is hurt on the hedge leg; the collar is roughly
neutral. State which side of it your strategy is on.

## The strategy library

Five standard strategies, each buildable from one option chain:
[OptionsPlaybook](https://www.optionsplaybook.com/option-strategies/)'s four *Rookie* strategies plus
one *Veteran*, the long call. The right-hand column is inspiration, not the answer; finding and
defending the pairing is the challenge.

**New to options?** Read *Options 101* below first. **Already trade them?** Skip to *What you submit*.

| # | Strategy | Tier | Outlook | Legs | An 8-K signal a team might test |
|---|---|---|---|---|---|
| 1 | Long call | Veteran | Bullish | Buy call | Guidance raised, acquisition completed, large contract award |
| 2 | Covered call | Rookie | Neutral to mildly bullish | Own stock, sell call | Dividend increase or buyback in a name you already hold |
| 3 | Protective put | Rookie | Hedge a long | Own stock, buy put | Regulatory investigation or material litigation disclosed |
| 4 | Collar | Rookie | Hedge, financed | Own stock, buy put, sell call | Holding through a restructuring or workforce reduction |
| 5 | Cash-secured put | Rookie | Bullish, paid to wait | Sell put, hold cash | Share repurchase program announced |

Whichever you pick, price the legs the way sections 5 and 6 do: from the chain as it existed on
the pre-event session, at the strikes your rule names, marked from daily option bars. Where a
strategy needs 100 shares, use the synthetic stock from section 6.

## Options 101 — the five strategies

If you have never traded an option, start here. Payoffs, break-evens and timing rules follow
[OptionsPlaybook](https://www.optionsplaybook.com/option-strategies/).

### The five at a glance

Numbered in the order they build on each other.

| # | Strategy | Tier | Outlook | Legs | You pay / are paid |
|---|---|---|---|---|---|
| 1 | Long call | Veteran | Up, and soon | Buy 1 ATM call | Debit |
| 2 | Covered call | Rookie | Flat to mildly up | Own 100 shares, sell 1 call | Credit |
| 3 | Protective put | Rookie | Up, but nervous | Own 100 shares, buy 1 put | Debit |
| 4 | Collar | Rookie | Up, but nervous, cheaply | Own 100 shares, buy put A, sell call B | Either, often ~zero |
| 5 | Cash-secured put | Rookie | Flat to up, happy to own it | Sell 1 put, set the cash aside | Credit |

**1** teaches what a call costs and what a break-even is, **2** is the other side of that contract,
**3** buys insurance instead of selling it, **4** is 2 and 3 at once, and **5** is 2's payoff without
owning the shares first.

---

### 1 · Long call

**What it is.** Buy one call at strike A, at the money on the pre-event session. You get the upside
of 100 shares for a fraction of their price, and can lose only what you paid.

**Construction** Buy 1 call, strike A ≈ spot · **Max gain** unlimited · **Max loss** the premium
paid · **Break-even** A + premium.

**The 8-K signal.** Guidance raised, an acquisition completed, a large contract award, a buyback
authorized — categories where the disclosure is unambiguously good news *and* you can argue the
move was not already in the price.

**Watch out for.** The move must arrive before expiry and be bigger than the premium. A stock that
rises 2% when you paid 4% for the call is a loss: compare the realized move against the implied
move, never against zero.

### 2 · Covered call

**What it is.** Own 100 shares, sell one call above the current price, keep the premium. If the
stock stays below the strike you keep everything; above it your shares are called away at the strike.
You have sold someone else strategy 1.

**Construction** Own 100 shares, sell 1 call at strike A above spot · **Max gain** (A − stock
price) + premium received · **Max loss** the stock's own downside, cushioned by the premium ·
**Break-even** stock price − premium. OptionsPlaybook's timing rule is to sell 30–45 days to expiry,
where time decay accelerates: the `1m` bucket.

**The 8-K signal.** A dividend increase or buyback in a name you hold; more generally, any category
where you can show the market over-pays for the implied move. That over-payment is the premium you
collect.

**Watch out for.** Opportunity risk in the tail: a merger 8-K that gaps the stock 30% above your
strike is the disaster case, and a category you can exclude deliberately. Mind early assignment on
the short call around dividends.

### 3 · Protective put

**What it is.** Own 100 shares, buy a put at strike A. Below A you are protected; above it you keep
the upside. It is insurance: the premium is the cost, the gap from spot down to A is the deductible.
Unlike a stop order it cannot be triggered by noise or gapped through overnight, which matters for
8-Ks, where the move often arrives as a gap.

**Construction** Own 100 shares, buy 1 put at strike A · **Max gain** unlimited, the upside is
uncapped · **Max loss** (stock price − A) + premium paid · **Break-even** stock price + premium.
You want IV to *rise* after you are in.

**The 8-K signal.** A regulatory investigation opened, material litigation disclosed, a
cybersecurity incident, an impairment — bad-but-ambiguous news where you want to stay long the name
and cap the left tail.

**Watch out for.** Insurance paid on every event is a real drag. The test is not "did the put pay
off on the bad events" but "what did the programme cost across all events in the category".

### 4 · Collar

**What it is.** Own the shares, buy a put at A, sell a call at B to fund it. A **zero-cost collar**
chooses B so the call premium offsets the put. The stock is boxed into the range A to B.

**Construction** Own 100 shares, buy put A, sell call B, A < spot < B · **Max gain** (B − stock
price), adjusted for the net debit or credit · **Max loss** (stock price − A), adjusted the same way
· **Break-even** stock price + net debit, or stock price − net credit.

**The 8-K signal.** Holding through a restructuring, a workforce reduction, a strategic review — an
event you must sit through, where you will trade away the upside tail to remove the downside one.

**Watch out for.** Two strikes, so your rule must fix both. The zero-cost constraint is the
cleanest way: it picks B once you have chosen A. "What deductible could you buy for free, on average,
ahead of this category?" is one number, novel, and defensible.

### 5 · Cash-secured put

**What it is.** Sell a put at a strike you would be glad to buy the stock at, and set aside the
cash to buy it. Same payoff shape as the covered call, without owning the shares first.

**Construction** Sell 1 put at strike A, hold A × 100 in cash · **Max gain** the premium received ·
**Max loss** substantial — down to A minus the premium, if the stock goes to zero ·
**Break-even** A − premium.

**The 8-K signal.** A share repurchase programme announced, an insider purchase, a restructuring you
read as a floor — cases where a dip is your entry rather than your risk.

**Watch out for.** The largest risk of the five. It is an obligation: an 8-K that discloses a
fraud hands you the shares all the way down, and you can be assigned early. If you run this, the
write-up must show sizing and the worst single event, not just the average.

---

### Picking one

Choose on the shape of your thesis:

- **"This category moves stocks more than the market expects, and I can call the direction."** →
  long call (1). Defending the sign is a strong claim; worth real credit if it survives the sealed
  window.
- **"This category is over-priced; the market pays for drama that does not arrive."** → covered
  call (2) or cash-secured put (5). The most novel direction here, and the one that most needs the
  placebo in section 7 to rule out "you were just short volatility in a calm year".
- **"You have to hold this name through the event."** → protective put (3) or collar (4). The
  question becomes what the protection costs, priced across the whole category.

Whichever you pick: price every leg from the chain as it existed on the pre-event session, enter at
the close of `t_0` (or `t_pre` if the question is whether the market priced it), exit at the fixed
horizons, use the synthetic stock where the strategy needs shares, and compare against ordinary days
for the same names.

## What you submit

1. **This notebook, or one built on it**, that runs from a clean kernel with only an API key. Your
   pipeline must take a start date and an end date as inputs; judges will call it on a window
   you have not seen.
2. **A write-up of at most two pages** with five parts: the hypothesis (which categories, which
   strategy, why the market should misprice it), the method, the result with uncertainty at
   every fixed horizon in-sample and out-of-sample, what would make it break, and how you would
   trade it.
3. **A parameter-sensitivity check**: the result across the neighbouring choices you could have
   made (expiry bucket, OTM distance, entry session, horizon, category definition), using the
   organizers' overfitting library or your own grid.

| Criterion | Weight | Full marks look like |
|---|---|---|
| Hypothesis and novelty | 30 | A non-obvious, economically motivated category-to-strategy link. "Guidance up, buy a call" is the floor. |
| Analytical rigor | 30 | All fixed horizons reported, a placebo or baseline, an out-of-sample window, uncertainty shown, sensitivity examined. |
| Sealed-window replication | 20 | The finding holds on the judges' window, or you predicted its fragility and were right. |
| Trade realism | 10 | Entry timing that respects the filing lag, costs and liquidity from the data, stated capacity. |
| Communication | 10 | A write-up a portfolio manager could act on; a notebook that runs. |

## Requirements and setup

Tested with the versions below; newer releases are fine, older major versions are not supported.

| Install | Tested version | Minimum | Why |
|---|---|---|---|
| **Python** | 3.14.3 | 3.10 | The code uses type-hint syntax that needs 3.10 or later. Get it from [python.org](https://www.python.org/downloads/) |
| **pandas** | 3.0.5 | 2.2 | Every table in the notebook |
| **numpy** | 2.5.2 | 2.0 | Bootstrap intervals and arithmetic |
| **requests** | 2.34.2 | 2.31 | Calls to the Massive REST API |
| **matplotlib** | 3.11.1 | 3.8 | Charts |
| **ipykernel** | 7.3.0 | 6.29 | Runs the notebook inside your virtual environment |
| **JupyterLab** or **VS Code** with the Jupyter extension | latest | | Opens and runs the notebook |

You also need a **Massive API key** with access to options data and Filings & Disclosures. Copy it
from the Discord channel.

### 1. Run the setup script

Open a terminal in the folder that holds this notebook and run:

```bash
./setup.sh                                            # macOS / Linux
powershell -ExecutionPolicy Bypass -File setup.ps1    # Windows
```

It creates a `.venv`, installs `requirements.txt`, registers the kernel and creates `.env` from
`.env.example`. It is safe to re-run. To do the same by hand:

```bash
python3 -m venv .venv
source .venv/bin/activate          # Windows: .venv\Scripts\activate
python -m pip install --upgrade pip
python -m pip install -r requirements.txt
python -m ipykernel install --user --name gator-quant-hacks --display-name "Python (Gator Quant Hacks .venv)"
```

Then open the notebook (`jupyter lab`, or open it in VS Code) and choose the kernel
**Python (Gator Quant Hacks .venv)**. Check that it worked by running `import sys; print(sys.executable)`
in a cell: the path should end in `.venv/bin/python`.

**Already in a Jupyter environment** (Colab, an existing kernel)? Skip the above and run the
install cell just below, then restart the kernel.

### 2. Put your API key in a `.env` file

The notebook looks for the key in this order: the `MASSIVE_API_KEY` environment variable, then a
file named `.env` **in the same folder as this notebook**, then a password prompt.

1. In the notebook's folder, create a file called `.env` (or copy `.env.example` and rename it).
2. Put one line in it, with your own key and no spaces or quotes:

   ```
   MASSIVE_API_KEY=your-key-here
   ```
3. Save it and restart the kernel. Section 1 prints `API key loaded (ends xxxx)` when it finds the key.

Your folder should look like this:

```
your-project/
├── .venv/                       ← the virtual environment
├── .env                         ← your API key (never share or commit this)
├── requirements.txt
├── setup.sh / setup.ps1
└── gator-quant-hacks-8k-options-challenge.ipynb
```

**Prefer an environment variable?** Run `export MASSIVE_API_KEY=your-key-here` in the terminal
(Windows PowerShell: `$env:MASSIVE_API_KEY="your-key-here"`) before starting Jupyter, from the same
terminal where the virtual environment is active.

**Keep the key out of your submission.** Never paste it into a notebook cell, and add `.env` to your
`.gitignore` if you use Git. Judges run your notebook with their own key.

In [ ]:
# Optional: only if you did NOT run setup.sh / setup.ps1. Installs into this kernel; restart the kernel afterwards.
# %pip install -q -r requirements.txt

## 1 · API client

Plain `requests` against the REST API. `api_get` caches every response on disk by URL and retries
rate limits and server errors; `api_get_all` follows `next_url` pagination. A full run is roughly
6,500 requests and takes about 10 minutes on a fresh cache (most of it the placebo), seconds on a
warm one.

In [ ]:
import os, json, time, hashlib, re
from dataclasses import dataclass, field
from pathlib import Path
from getpass import getpass

import numpy as np
import pandas as pd
import requests
import matplotlib.pyplot as plt
from matplotlib.ticker import PercentFormatter
from pandas.tseries.holiday import (AbstractHolidayCalendar, Holiday, nearest_workday, USMartinLutherKingJr,
                                    USPresidentsDay, GoodFriday, USMemorialDay, USLaborDay, USThanksgivingDay)
from pandas.tseries.offsets import CustomBusinessDay

BASE_URL = "https://api.massive.com"


def load_api_key(name: str = "MASSIVE_API_KEY") -> str:
    """Environment first, then a `.env` file beside the notebook, then a prompt. Never paste a key into a cell."""
    key = (os.environ.get(name) or "").strip()
    env_file = Path(".env")
    if not key and env_file.exists():
        for line in env_file.read_text().splitlines():
            if line.strip().startswith(f"{name}="):
                key = line.split("=", 1)[1].strip().strip('"').strip("'")
    if not key:
        key = getpass(f"{name} (https://massive.com/dashboard/keys): ").strip()
    return key


API_KEY = load_api_key()
SESSION = requests.Session()
SESSION.headers["Authorization"] = f"Bearer {API_KEY}"

CACHE_DIR = Path(".massive_cache")
CACHE_DIR.mkdir(exist_ok=True)


def api_get(path_or_url: str, params: dict | None = None) -> dict:
    """GET one page from the Massive REST API. Cached on disk by the full URL, so reruns are free."""
    url = path_or_url if path_or_url.startswith("http") else BASE_URL + path_or_url
    full_url = requests.Request("GET", url, params=params).prepare().url
    cache_file = CACHE_DIR / (hashlib.sha1(full_url.encode()).hexdigest() + ".json")
    if cache_file.exists():
        return json.loads(cache_file.read_text())
    for attempt in range(10):
        resp = SESSION.get(full_url, timeout=60)
        if resp.status_code in (429, 500, 502, 503, 504):   # rate-limited or a transient server error
            retry_after = resp.headers.get("Retry-After", "")
            time.sleep(float(retry_after) if retry_after.isdigit() else min(2 ** attempt, 20))
            continue
        break
    resp.raise_for_status()
    payload = resp.json()
    cache_file.write_text(json.dumps(payload))
    return payload


def api_get_all(path: str, params: dict | None = None, max_pages: int = 500) -> list[dict]:
    """Follow `next_url` pagination and return every result row."""
    payload = api_get(path, params)
    rows = list(payload.get("results") or [])
    pages = 1
    while payload.get("next_url") and pages < max_pages:
        payload = api_get(payload["next_url"])
        rows.extend(payload.get("results") or [])
        pages += 1
    return rows


print(f"API key loaded (ends {API_KEY[-4:]}); cache at {CACHE_DIR.resolve()}")

## 2 · Configuration

Everything a team is allowed to change lives in this cell. The horizons are **fixed for every
team** so results are comparable across submissions.

- **`EVENT_TAG`** is one `tertiary_category` from the taxonomy (next cell). `cfo_appointment` is the
  worked example; combine tags by building your own event table in section 4.
- **Three windows.** In-sample is where you research. Out-of-sample is a recent window the in-sample
  analysis never touches, run in this notebook; horizons that have not yet resolved for its latest
  events are absent. The sealed window belongs to the judges and is not run here.
- **`TOP_100`** is a static list of the 100 largest US companies. There is no point-in-time membership
  feed; the appendix names the cost.
- **`EXPIRY_BUCKETS`** are calendar days to expiry from the pre-event session. The headline bucket is
  3–6 months out, so that the out-of-sample test is on contracts that live long enough to be judged
  at every horizon.
- **`OTM_PCT`** is how far out of the money the sold call (covered call, collar) and the bought or
  sold put (protective put, collar, cash-secured put) sit. `OTM_GRID` is the neighbourhood the
  sensitivity section walks.
- **`ENTRY`**. `"post"` enters at the close of the filing session, which is the rule you could
  actually trade. `"pre"` enters the session before, which nobody could trade but which answers
  "did the market price this event?" Both are computed; this switch picks which one the tables show.

In [ ]:
# ---- The 8-K event. One tertiary_category tag from the taxonomy (printed in the next section). ------
EVENT_TAG = "cfo_appointment"

# ---- Windows. In-sample is what you research on; out-of-sample is the most recent six months for
#      which every fixed horizon has resolved; the sealed window is the judges' and is not run here.
STUDY_START, STUDY_END = "2024-01-01", "2025-12-31"      # in-sample (plan has ~2 years of options history)
OOS_START, OOS_END = "2026-01-01", "2026-08-31"          # out-of-sample, run in this notebook
HOLDOUT_START, HOLDOUT_END = "2023-06-01", "2023-08-31"  # sealed: judges change these and flip RUN_HOLDOUT

# ---- Universe: the 100 largest US companies by market value (approximately the S&P 100), one
#      ticker per company, as of September 2026. A static list, so mind the caveat in the appendix.
TOP_100 = """
AAPL ABBV ABT ACN ADBE AIG AMD AMGN AMT AMZN AVGO AXP BA BAC BK BKNG BLK BMY BRK.B C
CAT CHTR CL CMCSA COF COP COST CRM CSCO CVS CVX DE DHR DIS DUK EMR FDX GD GE GILD
GM GOOGL GS HD HON IBM INTC INTU ISRG JNJ JPM KO LIN LLY LMT LOW MA MCD MDLZ MDT
MET META MMM MO MRK MS MSFT NEE NFLX NKE NOW NVDA ORCL PEP PFE PG PLTR PM PYPL QCOM
RTX SBUX SCHW SO T TGT TMO TMUS TSLA TXN UBER UNH UNP UPS USB V VZ WFC WMT XOM
""".split()
assert len(TOP_100) == 100 and len(set(TOP_100)) == 100

# ---- Expiry buckets: name -> (min, max, target) calendar days to expiry from the pre-event session.
EXPIRY_BUCKETS = {
    "1m":   (21, 45, 30),
    "2m":   (46, 80, 60),
    "3-6m": (90, 180, 120),
}
BASELINE_BUCKET = "3-6m"     # the headline bucket: the out-of-sample test is on options 3-6 months out

# ---- Fixed horizons (sessions after the filing session). Fixed for every team. Do not change. -----
HORIZONS = [1, 2, 3, 5, 10, 21, 42, 63]

# ---- Strategy parameters -----------------------------------------------------------------------
OTM_PCT = 0.05              # covered call / collar sell the call ~5% above spot; protective put / collar / cash-secured put use the put ~5% below
OTM_GRID = [0.03, 0.05, 0.10]   # neighbours for the parameter-sensitivity check; OTM_PCT must be one of them
ENTRY = "post"              # "post": close of the filing session (the tradeable rule). "pre": close of the session before (hypothetical: was it priced?)
RISK_FREE = 0.04            # flat carry rate used in put-call parity to recover spot from the chain

# ---- Mechanics -----------------------------------------------------------------------------------
STRIKE_WINDOW = 0.25        # keep strikes within ±25% of spot once spot is known
MAX_STALE_SESSIONS = 3      # a leg's last trade may be at most this many sessions old to count as a mark
MAX_EVENTS = None           # e.g. 15 for a smoke test; None = all events

# ---- Stretch toggles -----------------------------------------------------------------------------
RUN_PLACEBO = True          # the same measurement on ordinary days for the same names
N_PLACEBO = 120
PLACEBO_GAP_DAYS = 30
FETCH_ACCEPTANCE_TIMES = True   # pull <ACCEPTANCE-DATETIME> from EDGAR to fix same-day vs next-day entry
SEC_USER_AGENT = "GatorQuantHacks team-name your@email.edu"   # the SEC requires a contact in the User-Agent
RUN_HOLDOUT = False         # judges flip this

assert OTM_PCT in OTM_GRID and BASELINE_BUCKET in EXPIRY_BUCKETS and ENTRY in ("pre", "post")

# ---- Catch impossible dates (e.g. "2024-06-31") here rather than deep in the pipeline -----------
for _name in ("STUDY_START", "STUDY_END", "OOS_START", "OOS_END", "HOLDOUT_START", "HOLDOUT_END"):
    try:
        pd.Timestamp(globals()[_name])
    except ValueError:
        raise ValueError(f"{_name} = {globals()[_name]!r} is not a real date (check the day of the month)") from None
assert STUDY_START < STUDY_END and OOS_START < OOS_END and HOLDOUT_START < HOLDOUT_END, "each window's start must be before its end"

### The disclosure taxonomy

One request returns the full three-level taxonomy. Anything in the `tertiary_category` column can
be an `EVENT_TAG`. The cell prints the family the chosen tag belongs to; drop the filter to see all 119.

In [ ]:
taxonomy = pd.DataFrame(api_get_all("/stocks/taxonomies/vX/disclosures", {"limit": 1000}))
print(f"{len(taxonomy)} tertiary tags, {taxonomy.secondary_category.nunique()} secondary, "
      f"{taxonomy.primary_category.nunique()} primary")
family = taxonomy.loc[taxonomy.tertiary_category == EVENT_TAG, "secondary_category"].iloc[0]
with pd.option_context("display.max_colwidth", 100):
    display(taxonomy[taxonomy.secondary_category == family]
            [["primary_category", "secondary_category", "tertiary_category", "description"]]
            .reset_index(drop=True))

## 3 · A trading calendar without a stock feed

Every horizon is counted in sessions, so we need to know which days the market was open. With no
stock bars to read it from, we build it: weekdays minus the NYSE full-day closures. NYSE differs from
the federal calendar in three places (Good Friday closed, Columbus Day and Veterans Day open, no
observance when New Year's Day falls on a Saturday) and had one special closure in this window, the
national day of mourning for President Carter on 9 January 2025. Section 5 checks this calendar
against the days options actually traded.

In [ ]:
class NYSEHolidays(AbstractHolidayCalendar):
    """NYSE full-day closures. Differs from the federal calendar: Good Friday closed, Columbus and
    Veterans Day open, no observance when New Year's Day falls on a Saturday."""
    rules = [
        Holiday("New Year's Day", month=1, day=1,
                observance=lambda d: d + pd.Timedelta(days=1) if d.weekday() == 6 else d),
        USMartinLutherKingJr, USPresidentsDay, GoodFriday, USMemorialDay,
        Holiday("Juneteenth", month=6, day=19, start_date="2022-01-01", observance=nearest_workday),
        Holiday("Independence Day", month=7, day=4, observance=nearest_workday),
        USLaborDay, USThanksgivingDay,
        Holiday("Christmas Day", month=12, day=25, observance=nearest_workday),
        Holiday("National day of mourning, President Carter", year=2025, month=1, day=9),
    ]


def trading_sessions(start, end) -> pd.DatetimeIndex:
    holidays = NYSEHolidays().holidays(pd.Timestamp(start) - pd.Timedelta(days=7), pd.Timestamp(end) + pd.Timedelta(days=7))
    return pd.bdate_range(start, end, freq=CustomBusinessDay(holidays=holidays))


CAL = trading_sessions("2021-06-01", "2027-12-31")
TODAY = pd.Timestamp.today().normalize()
LAST_SESSION = CAL[CAL.searchsorted(TODAY, side="right") - 1]


def session_on_or_after(day) -> pd.Timestamp:
    return CAL[CAL.searchsorted(pd.Timestamp(day), side="left")]


def session_before(day) -> pd.Timestamp:
    return CAL[CAL.searchsorted(pd.Timestamp(day), side="left") - 1]


def sessions_between(a, b) -> int:
    """Number of sessions strictly after `a` up to and including `b`."""
    return int(CAL.searchsorted(pd.Timestamp(b), side="right") - CAL.searchsorted(pd.Timestamp(a), side="right"))


print(f"{len(CAL)} sessions in the calendar; last completed session {LAST_SESSION.date()}. "
      f"Sessions per year: " + ", ".join(f"{y}: {((CAL.year == y)).sum()}" for y in range(2022, 2027)))

## 4 · Events

One request per tag, paginated, across **all** filers, then filtered to the universe. A single filing
can carry the same tag for two tickers (share classes), so we collapse to **one event per filer per
filing date**.

Two dates matter for everything downstream:

- **`t_0`, the filing session**: the first session on or after `filing_date`. Entering at its close
  is the tradeable rule. `filing_date` is the EDGAR submission date and has no time of day; a filing
  accepted after 16:00 ET could not be traded until the next session, and the timing stretch near the
  end fixes that with EDGAR's acceptance timestamps.
- **`t_pre`, the pre-event session**: the session before `t_0`. The option chain as it stood at
  this close cannot know the news, so it is where the implied move is read and where `"pre"` entry
  happens.

Companies have four business days to file after the event. `supporting_text` often names the
announcement or effective date; aligning to it rather than the filing date is a refinement judges
will credit.

In [ ]:
def normalize_ticker(t) -> str | None:
    """Filings write share classes as BRK/B or BRK.B; Massive's options use BRK.B as the underlying."""
    if not isinstance(t, str) or not t.strip():
        return None
    return t.strip().upper().replace("/", ".")


def fetch_disclosures(tag: str, start: str, end: str) -> pd.DataFrame:
    rows = api_get_all("/stocks/filings/8-K/vX/disclosures", {
        "tertiary_category": tag, "filing_date.gte": start, "filing_date.lte": end,
        "limit": 1000, "sort": "filing_date.asc",
    })
    df = pd.DataFrame(rows)
    if not df.empty:
        df["filing_date"] = pd.to_datetime(df["filing_date"])
    return df


def build_events(tag: str, start: str, end: str, universe: list[str]) -> pd.DataFrame:
    """One row per (filer, filing date) in the universe, with the filing session and the session before it."""
    raw = fetch_disclosures(tag, start, end)
    print(f"{len(raw):,} '{tag}' disclosures across all filers, {start}..{end}")
    ex = raw.explode("tickers").rename(columns={"tickers": "ticker"})
    ex["ticker"] = ex["ticker"].map(normalize_ticker)
    ex = ex[ex["ticker"].isin(universe)]
    ev = (ex.sort_values(["cik", "filing_date"])
            .groupby(["cik", "filing_date"], as_index=False)
            .agg(ticker=("ticker", "first"), accession_number=("accession_number", "first"),
                 filing_url=("filing_url", "first"), supporting_text=("supporting_text", "first"))
            .sort_values("filing_date").reset_index(drop=True))
    ev["t_0"] = ev["filing_date"].map(session_on_or_after)      # the filing session: first session on or after the filing date
    ev["t_pre"] = ev["t_0"].map(session_before)                 # the session before it: the chain here cannot know the news
    ev["days_since_prev_event"] = ev.groupby("ticker")["filing_date"].diff().dt.days
    return ev


events = build_events(EVENT_TAG, STUDY_START, STUDY_END, TOP_100)
if MAX_EVENTS:
    events = events.head(MAX_EVENTS).copy()
print(f"{len(events)} in-sample events across {events.ticker.nunique()} of the top 100; "
      f"{(events.filing_date != events.t_0).sum()} were filed on a non-session day (weekend or holiday)")
display(events.groupby(events.filing_date.dt.year).size().rename("events").to_frame().T)
with pd.option_context("display.max_colwidth", 110):
    display(events[["ticker", "filing_date", "t_pre", "t_0", "supporting_text"]].sample(5, random_state=1))

## 5 · The option chain, and the stock price hidden inside it

For each event, one request to the contracts endpoint with `as_of = t_pre` returns every contract
that existed that day with an expiry up to 180 days out. Then:

1. **Recover spot.** We have no stock price, but at any strike $K$ with both a call and a put,
   put-call parity says $S = K e^{-rT} + C - P$. The estimate is best where both legs are liquid,
   which is at the money, so `locate_spot` starts at the median strike of the nearest weekly expiry
   (new weeklies are listed around the current price, so that guess is already close), reads the
   pair's last closes, computes $S$, jumps to the strike nearest that estimate and repeats until the
   strike stops moving. Two or three steps, four to six requests, and the answer is typically within
   half a percent, even the day after a 50% crash.
2. **Pick the expiry** in each bucket: the listed expiry nearest the bucket's target that has a real
   chain (at least three strikes with both legs).
3. **Pick the strikes.** $K$ is the paired strike nearest spot. For each OTM level in `OTM_GRID`, the
   call strike $U$ is the first at or above spot × (1 + pct) and the put strike $L$ the first at or
   below spot × (1 − pct).
4. **Pull every leg once, for its whole life.** One request per contract returns its daily closes
   and volumes from ten days before `t_pre` to expiry, so every later mark-to-market is a lookup, not
   a request. A leg that did not trade on a given session is marked at its last trade if that is at
   most `MAX_STALE_SESSIONS` old, otherwise it is missing for that session.

Daily bars are the last trade of the day, not the closing quote, and about a fifth of ATM legs on
S&P 500 names do not trade on a given day. On the 100 largest names the ATM and 5%-OTM legs trade
almost every session, which is one reason the universe is what it is. If your key includes options
quotes, the closing NBBO mid is a cleaner mark and fills in the spread; see section 11.

In [ ]:
def option_bars(opt_ticker: str, start, end) -> pd.DataFrame:
    """Daily bars for one contract: index = session, columns close and volume. Only sessions where it traded."""
    rows = api_get_all(f"/v2/aggs/ticker/{opt_ticker}/range/1/day/{pd.Timestamp(start):%Y-%m-%d}/{pd.Timestamp(end):%Y-%m-%d}",
                       {"adjusted": "false", "sort": "asc", "limit": 50000})
    if not rows:
        return pd.DataFrame(columns=["close", "volume"], index=pd.DatetimeIndex([], name="session"))
    idx = (pd.to_datetime([r["t"] for r in rows], unit="ms", utc=True)
             .tz_convert("America/New_York").normalize().tz_localize(None))
    return pd.DataFrame({"close": [float(r["c"]) for r in rows], "volume": [float(r.get("v") or 0) for r in rows]},
                        index=pd.DatetimeIndex(idx, name="session"))


def fetch_chain(ticker: str, as_of: pd.Timestamp, dte_lo: int, dte_hi: int) -> pd.DataFrame:
    """Every standard contract that existed on `as_of` with an expiry `dte_lo`..`dte_hi` days out."""
    rows = api_get_all("/v3/reference/options/contracts", {
        "underlying_ticker": ticker, "as_of": as_of.strftime("%Y-%m-%d"),
        "expiration_date.gte": (as_of + pd.Timedelta(days=dte_lo)).strftime("%Y-%m-%d"),
        "expiration_date.lte": (as_of + pd.Timedelta(days=dte_hi)).strftime("%Y-%m-%d"),
        "limit": 1000,
    })
    chain = pd.DataFrame(rows)
    if chain.empty:
        return chain
    if "shares_per_contract" in chain:
        chain = chain[chain["shares_per_contract"].fillna(100) == 100]      # drop post-split non-standard series
    chain = chain[["ticker", "contract_type", "strike_price", "expiration_date"]].copy()
    chain["expiration_date"] = pd.to_datetime(chain["expiration_date"])
    chain["dte"] = (chain["expiration_date"] - as_of).dt.days
    chain["strike_price"] = chain["strike_price"].astype(float)
    return chain.reset_index(drop=True)


def paired_strikes(e: pd.DataFrame) -> np.ndarray:
    both = e.groupby("strike_price")["contract_type"].nunique()
    return both[both == 2].index.to_numpy(dtype=float)


def contract(e: pd.DataFrame, strike: float, kind: str) -> str:
    return e[(e.strike_price == strike) & (e.contract_type == kind)]["ticker"].iloc[0]


def last_close_on_or_before(opt_ticker: str, day: pd.Timestamp, lookback_days: int = 7) -> float | None:
    bars = option_bars(opt_ticker, day - pd.Timedelta(days=lookback_days), day)
    return float(bars["close"].iloc[-1]) if len(bars) else None


def locate_spot(chain: pd.DataFrame, day: pd.Timestamp, max_iter: int = 8) -> dict | None:
    """Recover the stock price on `day` from the option chain alone, via put-call parity.

    At any strike K with both a call and a put, S = K·e^(−rT) + C − P. The strike that minimises |C − P|
    is at-the-money, and the estimate is best there because both legs are liquid. We start at the
    median strike of the nearest expiry (new weekly series are listed around the current price, so that
    median is already close), then step to the strike nearest our estimate until it stops moving.
    """
    near = chain[chain.dte >= 3]
    if near.empty:
        return None
    near = near[near.dte == near.dte.min()]
    strikes = paired_strikes(near)
    if len(strikes) < 3:
        return None
    T = near.dte.iloc[0] / 365
    k, tried, est = float(np.median(strikes)), set(), None
    k = strikes[np.abs(strikes - k).argmin()]
    for _ in range(max_iter):
        tried.add(k)
        c = last_close_on_or_before(contract(near, k, "call"), day)
        p = last_close_on_or_before(contract(near, k, "put"), day)
        if c is None or p is None:                                   # an illiquid strike: try the next-nearest untried one
            rest = [s for s in strikes if s not in tried]
            if not rest:
                break
            k = rest[int(np.abs(np.array(rest) - k).argmin())]
            continue
        est = k * np.exp(-RISK_FREE * T) + c - p
        k_new = strikes[np.abs(strikes - est).argmin()]
        if k_new == k or k_new in tried:
            break
        k = k_new
    if est is None:
        return None
    return {"spot": float(est), "strike": float(k), "expiry": near.expiration_date.iloc[0], "dte": int(near.dte.iloc[0])}


def pick_expiry(chain: pd.DataFrame, lo: int, hi: int, target: int) -> pd.Timestamp | None:
    cand = chain[(chain.dte >= lo) & (chain.dte <= hi)]
    if cand.empty:
        return None
    dte_of = cand.groupby("expiration_date")["dte"].first()
    ok = [x for x, g in cand.groupby("expiration_date") if len(paired_strikes(g)) >= 3]   # needs a real chain, not a stub
    if not ok:
        return None
    return min(ok, key=lambda x: abs(dte_of[x] - target))


def select_strikes(e: pd.DataFrame, spot: float, otm_pcts: list[float]) -> dict[str, float] | None:
    """ATM strike K (nearest to spot, both legs listed) plus, for each OTM level, the call strike U at or
    above spot·(1+pct) and the put strike L at or below spot·(1−pct)."""
    both = paired_strikes(e)
    both = both[(both >= spot * (1 - STRIKE_WINDOW)) & (both <= spot * (1 + STRIKE_WINDOW))]
    if len(both) == 0:
        return None
    calls = np.sort(e.loc[e.contract_type == "call", "strike_price"].unique())
    puts = np.sort(e.loc[e.contract_type == "put", "strike_price"].unique())
    out = {"K": float(both[np.abs(both - spot).argmin()])}
    for pct in otm_pcts:
        up, dn = calls[calls >= spot * (1 + pct)], puts[puts <= spot * (1 - pct)]
        out[f"U{pct}"] = float(up.min()) if len(up) else float(calls.max())
        out[f"L{pct}"] = float(dn.max()) if len(dn) else float(puts.min())
    return out

### Pricing every event

`price_events` runs the four steps above for each event and bucket and returns a list of
`PricedEvent` objects: the recovered spot, the expiry, the strikes and every leg's bar history.
`summarize_priced` flattens what you need for a first look. The `abs_spot_gap` column is a
self-check: the spot implied by this bucket's own ATM pair against the spot recovered from the
near-dated pair. They should agree to well under a percent.

In [ ]:
@dataclass
class Leg:
    ticker: str
    kind: str                 # "call" or "put"
    strike: float
    bars: pd.DataFrame        # close, volume by session (only sessions with trades)

    def mark(self, day: pd.Timestamp) -> float:
        """Last traded close on or before `day`, if it is at most MAX_STALE_SESSIONS sessions old."""
        b = self.bars.loc[: pd.Timestamp(day)]
        if b.empty or sessions_between(b.index[-1], day) > MAX_STALE_SESSIONS:
            return np.nan
        return float(b["close"].iloc[-1])

    def volume_on(self, day: pd.Timestamp) -> float:
        return float(self.bars["volume"].get(pd.Timestamp(day), 0.0))


@dataclass
class PricedEvent:
    """One event × one expiry bucket: the chain-derived spot, the expiry and every leg a strategy may need."""
    ticker: str
    event_date: pd.Timestamp
    t_pre: pd.Timestamp
    t_0: pd.Timestamp
    bucket: str
    expiry: pd.Timestamp
    expiry_session: pd.Timestamp        # last session on or before expiry
    spot_pre: float                     # chain-implied spot on t_pre
    strikes: dict[str, float]
    legs: dict[str, Leg]                # "C_K", "P_K", "C_U0.05", "P_L0.05", ...

    def marks(self, day) -> dict[str, float]:
        return {name: leg.mark(day) for name, leg in self.legs.items()}

    def synthetic_spot(self, day, m: dict[str, float] | None = None) -> float:
        """Stock price implied by the ATM pair on `day`: K·e^(−rT) + C_K − P_K. Exact at expiry."""
        m = m if m is not None else self.marks(day)
        T = max((self.expiry - pd.Timestamp(day)).days, 0) / 365
        return self.strikes["K"] * np.exp(-RISK_FREE * T) + m["C_K"] - m["P_K"]


def price_event(ticker: str, t_pre: pd.Timestamp, t_0: pd.Timestamp, event_date: pd.Timestamp,
                buckets: dict, otm_pcts: list[float]) -> tuple[list[PricedEvent], list[str]]:
    dte_hi = max(b[1] for b in buckets.values())
    chain = fetch_chain(ticker, t_pre, 2, dte_hi)
    if chain.empty:
        return [], ["no option chain as of the pre-event session"]
    loc = locate_spot(chain, t_pre)
    if loc is None:
        return [], ["could not recover spot from the chain (no liquid near-dated pair)"]
    priced, notes = [], []
    for name, (lo, hi, target) in buckets.items():
        expiry = pick_expiry(chain, lo, hi, target)
        if expiry is None:
            notes.append(f"{name}: no expiry {lo}-{hi} days out")
            continue
        e = chain[chain.expiration_date == expiry]
        strikes = select_strikes(e, loc["spot"], otm_pcts)
        if strikes is None:
            notes.append(f"{name}: no paired strikes near spot")
            continue
        wanted = {"C_K": ("call", strikes["K"]), "P_K": ("put", strikes["K"])}
        for pct in otm_pcts:
            wanted[f"C_U{pct}"] = ("call", strikes[f"U{pct}"])
            wanted[f"P_L{pct}"] = ("put", strikes[f"L{pct}"])
        legs = {}
        for key, (kind, k) in wanted.items():
            tk = contract(e, k, kind)
            legs[key] = Leg(tk, kind, k, option_bars(tk, t_pre - pd.Timedelta(days=10), expiry))
        pe = PricedEvent(ticker, event_date, t_pre, t_0, name, expiry, CAL[CAL.searchsorted(expiry, side="right") - 1],
                         loc["spot"], strikes, legs)
        if np.isnan(pe.legs["C_K"].mark(t_pre)) or np.isnan(pe.legs["P_K"].mark(t_pre)):
            notes.append(f"{name}: ATM pair did not trade on or near the pre-event session")
            continue
        priced.append(pe)
    return priced, notes


def price_events(ev: pd.DataFrame, buckets: dict = None, otm_pcts: list[float] = None, label: str = "events") -> tuple[list[PricedEvent], pd.DataFrame]:
    buckets = buckets or EXPIRY_BUCKETS
    otm_pcts = otm_pcts or OTM_GRID
    priced, dropped, t0 = [], [], time.time()
    for n, row in enumerate(ev.itertuples(index=False), 1):
        got, notes = price_event(row.ticker, row.t_pre, row.t_0, row.event_date if hasattr(row, "event_date") else row.filing_date,
                                 buckets, otm_pcts)
        priced += got
        dropped += [(row.ticker, row.t_0, note) for note in notes]
        if n % 25 == 0:
            print(f"  {label}: {n}/{len(ev)} events, {time.time() - t0:.0f}s")
    print(f"{label}: {len(ev)} events -> {len(priced)} priced (event, bucket) pairs; {len(dropped)} drops; {time.time() - t0:.0f}s")
    return priced, pd.DataFrame(dropped, columns=["ticker", "t_0", "reason"])


def summarize_priced(priced: list[PricedEvent]) -> pd.DataFrame:
    rows = []
    for pe in priced:
        m = pe.marks(pe.t_pre)
        rows.append({"ticker": pe.ticker, "event_date": pe.event_date, "t_pre": pe.t_pre, "t_0": pe.t_0, "bucket": pe.bucket,
                     "expiry": pe.expiry, "dte": (pe.expiry - pe.t_pre).days, "spot_pre": pe.spot_pre, "K": pe.strikes["K"],
                     "moneyness": pe.strikes["K"] / pe.spot_pre - 1, "call": pe.legs["C_K"].ticker, "put": pe.legs["P_K"].ticker,
                     "call_px": m["C_K"], "put_px": m["P_K"], "implied_move": (m["C_K"] + m["P_K"]) / pe.spot_pre,
                     "spot_gap": pe.synthetic_spot(pe.t_pre, m) / pe.spot_pre - 1,     # this bucket's parity spot vs the near-dated one
                     "atm_volume": pe.legs["C_K"].volume_on(pe.t_pre) + pe.legs["P_K"].volume_on(pe.t_pre)})
    return pd.DataFrame(rows)


events["event_date"] = events["filing_date"]
priced, dropped = price_events(events, label="in-sample")
panel = summarize_priced(priced)
if len(dropped):
    display(dropped["reason"].value_counts().rename("dropped").to_frame())
display(panel.assign(abs_spot_gap=panel.spot_gap.abs()).groupby("bucket")[["dte", "moneyness", "abs_spot_gap", "implied_move", "atm_volume"]]
        .median().round(3).rename(columns=lambda c: f"median {c}"))
with pd.option_context("display.max_colwidth", 40):
    display(panel[panel.bucket == BASELINE_BUCKET][["ticker", "t_pre", "spot_pre", "K", "call", "put", "call_px", "put_px", "implied_move"]].head(6))

## 6 · The P&L engine: five strategies from four legs

Each strategy is a fixed combination of the legs already pulled, marked at the entry session and at
each exit session, and expressed as **P&L per $1 of stock at entry** so that a $60 stock and a $600
stock are comparable and so that the stock's own return sits on the same scale.

With $S$ the synthetic stock ($K e^{-rT} + C_K - P_K$ from the ATM pair), $C_K$ the ATM call,
$C_U$ the call at $U$ and $P_L$ the put at $L$, and $\Delta$ meaning "exit minus entry":

| Strategy | P&L per $1 of spot at entry |
|---|---|
| Stock only (reference) | $\Delta S / S_e$ |
| 1 · Long call | $\Delta C_K / S_e$ |
| 2 · Covered call | $(\Delta S - \Delta C_U) / S_e$ |
| 3 · Protective put | $(\Delta S + \Delta P_L) / S_e$ |
| 4 · Collar | $(\Delta S + \Delta P_L - \Delta C_U) / S_e$ |
| 5 · Cash-secured put | $-\Delta P_L / S_e$ |

The stock leg is the synthetic long, not shares. That is exactly right for a challenge with no stock
data, and nearly right in general: the synthetic does not collect dividends and cannot be assigned
early, which the appendix discusses.

`evaluate` builds one long table with a row per (event, bucket, entry, OTM level, horizon): the
fixed horizons are sessions after `t_0`, `"exp"` is the last session on or before expiry, and
horizons that fall after expiry or after the last completed session are simply absent. Everything
from here on is a group-by on that table.

In [ ]:
STRATEGIES = ["stock", "long_call", "covered_call", "protective_put", "collar", "cash_secured_put"]
STRATEGY_LABEL = {"stock": "Stock only (synthetic)", "long_call": "1 · Long call", "covered_call": "2 · Covered call",
                  "protective_put": "3 · Protective put", "collar": "4 · Collar", "cash_secured_put": "5 · Cash-secured put"}


def strategy_pnl(m_e: dict, m_x: dict, S_e: float, S_x: float, otm: float) -> dict[str, float]:
    """P&L per $1 of stock at entry, from entry marks `m_e` to exit marks `m_x`.

    The 100 shares behind the covered call, protective put and collar are replaced by the synthetic
    stock (long ATM call, short ATM put), which is what `S_e` and `S_x` are.
    """
    dS = S_x - S_e
    dC_U = m_x[f"C_U{otm}"] - m_e[f"C_U{otm}"]     # the OTM call we sell
    dP_L = m_x[f"P_L{otm}"] - m_e[f"P_L{otm}"]     # the OTM put we buy (or sell, cash-secured)
    dC_K = m_x["C_K"] - m_e["C_K"]                 # the ATM call we buy
    return {
        "stock":            dS / S_e,
        "long_call":        dC_K / S_e,
        "covered_call":     (dS - dC_U) / S_e,
        "protective_put":   (dS + dP_L) / S_e,
        "collar":           (dS + dP_L - dC_U) / S_e,
        "cash_secured_put": (-dP_L) / S_e,
    }


def evaluate(priced: list[PricedEvent], otm_pcts: list[float] = None) -> pd.DataFrame:
    """Long table: one row per (event, bucket, entry, OTM level, horizon) with every strategy's P&L."""
    otm_pcts = otm_pcts or OTM_GRID
    rows = []
    for pe in priced:
        i0 = CAL.get_loc(pe.t_0)
        exits = {0: pe.t_0}
        exits.update({h: CAL[i0 + h] for h in HORIZONS if i0 + h < len(CAL) and CAL[i0 + h] <= pe.expiry_session})
        exits["exp"] = pe.expiry_session
        for entry, e_day in (("pre", pe.t_pre), ("post", pe.t_0)):
            m_e = pe.marks(e_day)
            S_e = pe.synthetic_spot(e_day, m_e)
            if np.isnan(S_e):
                continue
            implied = (m_e["C_K"] + m_e["P_K"]) / S_e
            dte_sessions = sessions_between(e_day, pe.expiry_session)
            for h, x_day in exits.items():
                if x_day > LAST_SESSION:
                    continue
                m_x = pe.marks(x_day)
                S_x = pe.synthetic_spot(x_day, m_x)
                held = sessions_between(e_day, x_day)
                base = {"ticker": pe.ticker, "event_date": pe.event_date, "t_0": pe.t_0, "bucket": pe.bucket, "expiry": pe.expiry,
                        "entry": entry, "entry_date": e_day, "horizon": h, "exit_date": x_day, "sessions_held": held,
                        "dte_sessions": dte_sessions, "S_entry": S_e, "S_exit": S_x, "realized": S_x / S_e - 1,
                        "implied_move": implied,
                        "implied_scaled": implied * np.sqrt(held / dte_sessions) if dte_sessions else np.nan}
                for otm in otm_pcts:
                    rows.append(dict(base, otm=otm, **strategy_pnl(m_e, m_x, S_e, S_x, otm)))
    res = pd.DataFrame(rows)
    res["ratio"] = res["realized"].abs() / res["implied_scaled"]
    return res


results = evaluate(priced)
print(f"{len(results):,} rows: {results.ticker.nunique()} tickers × buckets × 2 entries × {len(OTM_GRID)} OTM levels × horizons")
hz = results[(results.bucket == BASELINE_BUCKET) & (results.entry == ENTRY) & (results.otm == OTM_PCT)]
display(hz.groupby("horizon", sort=False)[STRATEGIES].mean().mul(100).round(2).rename(columns=STRATEGY_LABEL)
        .rename_axis("mean P&L, % of spot at entry").T)

## 7 · Scoreboard

`scoreboard` reports, for each strategy and horizon, the mean P&L per $1 of spot with a 95%
bootstrap interval, the median and the share of events that made money. `fmt_board` shows the mean
and stars the cells whose interval excludes zero. Three things to hold in mind while reading it:

- **The stock row is the reference.** Three of the five strategies own the stock. If the stock row
  drives everything, the finding is about the stock after the event, not about the option.
- **Short options are paid on ordinary days too.** Options are usually a little overpriced (the
  variance risk premium), so a covered call or cash-secured put showing a positive mean is the
  default, not a finding. The placebo below measures that default for these same names.
- **Mean P&L is heavy-tailed** at these sample sizes. Always read the interval and the hit rate next
  to the mean, and open the filings behind the two or three largest events.

In [ ]:
def bootstrap_ci(x, n_boot: int = 2000, seed: int = 0) -> tuple[float, float]:
    x = np.asarray(x, dtype=float)
    x = x[~np.isnan(x)]
    if len(x) < 5:
        return (np.nan, np.nan)
    rng = np.random.default_rng(seed)
    means = rng.choice(x, size=(n_boot, len(x)), replace=True).mean(axis=1)
    return tuple(np.percentile(means, [2.5, 97.5]).tolist())


def slice_results(res: pd.DataFrame, bucket: str, entry: str, otm: float) -> pd.DataFrame:
    return res[(res.bucket == bucket) & (res.entry == entry) & (res.otm == otm)]


def scoreboard(res: pd.DataFrame, bucket: str = None, entry: str = None, otm: float = None,
               horizons=None, strategies=STRATEGIES) -> pd.DataFrame:
    """Rows: strategy. Columns: horizon. Cells: mean P&L per $1 of spot, with n and a 95% bootstrap CI."""
    r = slice_results(res, bucket or BASELINE_BUCKET, entry or ENTRY, otm or OTM_PCT)
    horizons = horizons or [h for h in HORIZONS + ["exp"] if h in set(r.horizon)]
    rows = []
    for s in strategies:
        for h in horizons:
            x = r.loc[r.horizon == h, s].dropna()
            lo, hi = bootstrap_ci(x)
            rows.append({"strategy": s, "horizon": h, "n": len(x), "mean": x.mean(), "ci_lo": lo, "ci_hi": hi,
                         "median": x.median(), "hit_rate": (x > 0).mean()})
    return pd.DataFrame(rows)


def difference_board(res_a: pd.DataFrame, res_b: pd.DataFrame, bucket: str = None, entry: str = None, otm: float = None,
                     strategies=STRATEGIES, seed: int = 1) -> pd.DataFrame:
    """Mean P&L of group a minus group b, per strategy and horizon, with a bootstrap CI on the difference."""
    a = slice_results(res_a, bucket or BASELINE_BUCKET, entry or ENTRY, otm or OTM_PCT)
    b = slice_results(res_b, bucket or BASELINE_BUCKET, entry or ENTRY, otm or OTM_PCT)
    rng, rows = np.random.default_rng(seed), []
    for s in strategies:
        for h in [h for h in HORIZONS + ["exp"] if h in set(a.horizon) & set(b.horizon)]:
            xa, xb = a.loc[a.horizon == h, s].dropna().to_numpy(), b.loc[b.horizon == h, s].dropna().to_numpy()
            row = {"strategy": s, "horizon": h, "n_a": len(xa), "n_b": len(xb)}
            if len(xa) >= 5 and len(xb) >= 5:
                d = rng.choice(xa, (2000, len(xa))).mean(1) - rng.choice(xb, (2000, len(xb))).mean(1)
                row.update(mean_a=xa.mean(), mean_b=xb.mean(), difference=xa.mean() - xb.mean(),
                           ci_lo=np.percentile(d, 2.5), ci_hi=np.percentile(d, 97.5))
            rows.append(row)
    return pd.DataFrame(rows)


def fmt_board(board: pd.DataFrame, value: str = "mean", star_if_ci_excludes_zero: bool = True) -> pd.DataFrame:
    """Wide display table: strategies × horizons, values in % of spot, * where the 95% CI excludes zero."""
    def cell(row):
        v = row[value]
        if pd.isna(v):
            return ""
        star = "*" if star_if_ci_excludes_zero and pd.notna(row.get("ci_lo")) and (row["ci_lo"] > 0 or row["ci_hi"] < 0) else ""
        return f"{v * 100:+.2f}%{star}"
    wide = board.assign(cell=board.apply(cell, axis=1)).pivot(index="strategy", columns="horizon", values="cell")
    wide = wide.reindex([s for s in STRATEGIES if s in wide.index]).rename(index=STRATEGY_LABEL)
    wide.columns = [f"h={c}" if c != "exp" else "expiry" for c in wide.columns]
    return wide.rename_axis(f"P&L per $1 spot", axis=0)


INK, INK2, MUTED, GRID, AXIS, SURFACE = "#0b0b0b", "#52514e", "#898781", "#e1e0d9", "#c3c2b7", "#fcfcfb"
SERIES = {"events": "#2a78d6", "placebo": "#eb6834", "oos": "#1baf7a", "in-sample": "#2a78d6"}


def style(ax, title: str, xlabel: str = "", ylabel: str = ""):
    ax.figure.set_facecolor(SURFACE)
    ax.set_facecolor(SURFACE)
    for s in ("top", "right"):
        ax.spines[s].set_visible(False)
    for s in ("left", "bottom"):
        ax.spines[s].set_color(AXIS)
        ax.spines[s].set_linewidth(0.8)
    ax.grid(axis="y", color=GRID, linewidth=0.8)
    ax.set_axisbelow(True)
    ax.tick_params(colors=INK2, labelsize=9, length=0)
    ax.set_title(title, loc="left", color=INK, fontsize=11, fontweight="bold", pad=10)
    ax.set_xlabel(xlabel, color=INK2, fontsize=9)
    ax.set_ylabel(ylabel, color=INK2, fontsize=9)


def plot_scoreboards(boards: dict[str, pd.DataFrame], title: str, strategies=STRATEGIES):
    """Small multiples: one panel per strategy, one line per group, band = 95% bootstrap CI of the first group."""
    horizons = [h for h in HORIZONS + ["exp"] if any(h in set(b.horizon) for b in boards.values())]
    x = np.arange(len(horizons))
    fig, axes = plt.subplots(2, 3, figsize=(13, 6.6), sharex=True)
    for ax, s in zip(axes.ravel(), strategies):
        for i, (name, b) in enumerate(boards.items()):
            t = b[b.strategy == s].set_index("horizon").reindex(horizons)
            c = SERIES.get(name, INK2)
            if i == 0:
                ax.fill_between(x, t.ci_lo.to_numpy(float) * 100, t.ci_hi.to_numpy(float) * 100, color=c, alpha=0.10, linewidth=0)
            ax.plot(x, t["mean"].to_numpy(float) * 100, color=c, linewidth=2, marker="o", markersize=5,
                    markeredgecolor=SURFACE, markeredgewidth=1.2, linestyle="-" if i == 0 else "--",
                    label=f"{name} (n={int(t.n.max())})")
        ax.axhline(0, color=MUTED, linewidth=0.8)
        ax.set_xticks(x, [str(h) if h != "exp" else "exp" for h in horizons])
        style(ax, STRATEGY_LABEL[s], "sessions after the filing session" if ax in axes[1] else "",
              "mean P&L, % of spot" if ax in axes[:, 0] else "")
    axes[0, 0].legend(frameon=False, labelcolor=INK2, fontsize=8, loc="upper left")
    fig.suptitle(title, x=0.01, ha="left", color=INK, fontsize=12, fontweight="bold")
    plt.tight_layout()
    plt.show()


board = scoreboard(results)
print(f"In-sample · {EVENT_TAG} · {BASELINE_BUCKET} options · entry = {ENTRY} · OTM {OTM_PCT:.0%} · "
      f"n = {int(board.n.max())} events. * = 95% bootstrap CI excludes zero.")
display(fmt_board(board))

### The placebo: the same names on ordinary days

The identical measurement on random sessions for the same tickers, drawn in proportion to how often
each ticker has an event and at least `PLACEBO_GAP_DAYS` from any of them. **The finding is the gap
between the two**, strategy by strategy, with its interval. A strategy that does as well on ordinary
days is a strategy, not a signal. This is the slowest cell (120 placebo events); set `RUN_PLACEBO =
False` for a quick iteration and back on before you submit.

In [ ]:
def sample_placebo(ev: pd.DataFrame, n: int, start: str, end: str, gap_days: int = PLACEBO_GAP_DAYS, seed: int = 7) -> pd.DataFrame:
    """Ordinary sessions for the same tickers, drawn in proportion to how often each ticker has an event,
    at least `gap_days` from any of that ticker's events."""
    rng = np.random.default_rng(seed)
    sessions = CAL[(CAL >= pd.Timestamp(start)) & (CAL <= pd.Timestamp(end))]
    by_ticker = ev.groupby("ticker")["filing_date"].apply(list)
    rows = []
    for t in rng.choice(ev["ticker"].to_numpy(), size=n, replace=True):
        for _ in range(25):
            d = sessions[rng.integers(len(sessions))]
            if all(abs((d - a).days) > gap_days for a in by_ticker.get(t, [])):
                rows.append({"ticker": t, "filing_date": d, "event_date": d, "t_0": d, "t_pre": session_before(d)})
                break
    return pd.DataFrame(rows)


if RUN_PLACEBO:
    placebo_events = sample_placebo(events, N_PLACEBO, STUDY_START, STUDY_END)
    placebo_priced, placebo_dropped = price_events(placebo_events, label="placebo")
    placebo_results = evaluate(placebo_priced)
    placebo_board = scoreboard(placebo_results)
    plot_scoreboards({"events": board, "placebo": placebo_board},
                     f"{EVENT_TAG} vs ordinary days for the same names · {BASELINE_BUCKET} options · entry = {ENTRY} · OTM {OTM_PCT:.0%}")
    diff = difference_board(results, placebo_results)
    print("Events minus ordinary days, mean P&L per $1 spot. * = the 95% interval on the difference excludes zero.")
    display(fmt_board(diff, value="difference"))

### The in-sample answer

`rank_strategies` orders the five by their event-minus-placebo edge, averaged over the 21-session,
42-session and expiry horizons of the headline bucket. That averaging rule is itself a choice; the
sensitivity section shows how much the answer depends on it. Whatever comes top here is the strategy
the out-of-sample section tests and the worked example walks through.

In [ ]:
def rank_strategies(diff: pd.DataFrame, horizons=(21, 42, "exp")) -> pd.DataFrame:
    """Rank the five strategies by their event-minus-placebo edge, averaged over the given horizons."""
    d = diff[diff.horizon.isin(horizons) & (diff.strategy != "stock")]
    out = (d.groupby("strategy").agg(edge=("difference", "mean"), horizons_ci_excludes_0=("ci_lo", lambda s: int(((s > 0) | (d.loc[s.index, "ci_hi"] < 0)).sum())),
                                     n_events=("n_a", "max"))
             .sort_values("edge", ascending=False))
    out["edge"] = out["edge"].map(lambda v: f"{v * 100:+.2f}%")
    return out.rename(index=STRATEGY_LABEL)


if RUN_PLACEBO:
    ranking = rank_strategies(diff)
    WINNER = [k for k, v in STRATEGY_LABEL.items() if v == ranking.index[0]][0]
    display(ranking)
    print(f"In-sample answer: after a '{EVENT_TAG}' 8-K, the strategy with the largest edge over ordinary days is "
          f"'{STRATEGY_LABEL[WINNER]}' ({BASELINE_BUCKET} options, entry = {ENTRY}, OTM {OTM_PCT:.0%}).")
else:
    WINNER = "long_call"

### The yardstick: did the market price the event?

The other use of the ATM pair. Its cost on `t_pre`, divided by spot, is the **implied move** to
expiry; the synthetic stock gives the realized move. For the fixed horizons we scale the implied
move with the square-root-of-time rule and report the ratio |realized| ÷ implied. A ratio near 1
means the market priced the event about right; well above 1 at short horizons means a jump the chain
did not see coming. Events and placebo side by side tell you whether this category is priced any
differently from ordinary days.

In [ ]:
def decay_table(r: pd.DataFrame) -> pd.DataFrame:
    rows = []
    for h in [h for h in HORIZONS + ["exp"] if h in set(r.horizon)]:
        x = r.loc[r.horizon == h, "ratio"].dropna()
        lo, hi = bootstrap_ci(x)
        rows.append({"horizon": h, "n": len(x), "mean_ratio": x.mean(), "ci_lo": lo, "ci_hi": hi,
                     "median_ratio": x.median(), "share_above_1": (x > 1).mean()})
    return pd.DataFrame(rows).set_index("horizon")


def fmt_decay(tbl: pd.DataFrame) -> pd.DataFrame:
    out = tbl.copy()
    for c in ["mean_ratio", "ci_lo", "ci_hi", "median_ratio"]:
        out[c] = out[c].map(lambda v: f"{v:.2f}")
    out["share_above_1"] = out["share_above_1"].map(lambda v: f"{v:.0%}")
    return out


pre = slice_results(results, BASELINE_BUCKET, "pre", OTM_PCT)
pre_exp = pre[pre.horizon == "exp"]
print(f"{BASELINE_BUCKET} ATM pair on the pre-event session: median implied move to expiry {pre_exp.implied_move.median():.1%}, "
      f"median |realized| {pre_exp.realized.abs().median():.1%}; realized beat implied in {(pre_exp.realized.abs() > pre_exp.implied_move).mean():.0%} of events")
tables = {"events": decay_table(pre)}
if RUN_PLACEBO:
    tables["placebo"] = decay_table(slice_results(placebo_results, BASELINE_BUCKET, "pre", OTM_PCT))
display(fmt_decay(pd.concat(tables, names=["group", "horizon"])))

## 8 · Out-of-sample

Nothing above touched these dates. The same pipeline runs on `OOS_START`..`OOS_END` (section 2), and
the comparison table puts in-sample and out-of-sample means side by side with a mark for whether the
sign agreed. Horizons that have not yet resolved for the latest events are absent, not zero.

Read it with the sample size in view. A dozen or two events can show whether the in-sample answer
had the right sign and rough magnitude, not more; saying so in the write-up scores better than
pretending otherwise.

This is *your* out-of-sample test. The judges' sealed window (final section) is a further period you
are told nothing about.

In [ ]:
oos_events = build_events(EVENT_TAG, OOS_START, OOS_END, TOP_100)
oos_events["event_date"] = oos_events["filing_date"]
print(f"{len(oos_events)} out-of-sample events across {oos_events.ticker.nunique()} tickers")
display(oos_events[["ticker", "filing_date", "t_pre", "t_0", "days_since_prev_event"]].reset_index(drop=True))
oos_priced, oos_dropped = price_events(oos_events, label="out-of-sample")
oos_results = evaluate(oos_priced)
oos_board = scoreboard(oos_results)
print(f"Out-of-sample · {OOS_START}..{OOS_END} · {BASELINE_BUCKET} options · entry = {ENTRY} · OTM {OTM_PCT:.0%} · n = {int(oos_board.n.max())}")
display(fmt_board(oos_board))

comparison = (board.merge(oos_board, on=["strategy", "horizon"], suffixes=("_in", "_oos"))
                   .assign(same_sign=lambda d: np.sign(d.mean_in) == np.sign(d.mean_oos)))
comp = comparison[comparison.horizon.isin([5, 21, 42, "exp"])]
comp_wide = (comp.assign(cell=lambda d: [f"{a * 100:+.2f}% → {b * 100:+.2f}% {'✓' if s else '✗'}"
                                        for a, b, s in zip(d.mean_in, d.mean_oos, d.same_sign)])
                 .pivot(index="strategy", columns="horizon", values="cell")
                 .reindex(STRATEGIES).rename(index=STRATEGY_LABEL))
comp_wide.columns = [f"h={c}" if c != "exp" else "expiry" for c in comp_wide.columns]
print(f"In-sample mean → out-of-sample mean (per $1 spot); ✓ = same sign. Out-of-sample n = {int(oos_board.n.max())}, "
      f"in-sample n = {int(board.n.max())}.")
display(comp_wide.rename_axis("in → out"))
plot_scoreboards({"in-sample": board, "oos": oos_board},
                 f"{EVENT_TAG}: in-sample ({STUDY_START[:7]}..{STUDY_END[:7]}) vs out-of-sample ({OOS_START[:7]}..{OOS_END[:7]}) · {BASELINE_BUCKET} options")

## 9 · One ticker, end to end

Only now do we pick a name. The rule is mechanical: among tickers with an out-of-sample event, the
one with the most in-sample events, ties broken by ATM volume (if no ticker has both, the busiest
in-sample name). The cell lists the contracts used for each event (real OCC symbols), every
strategy's P&L at four horizons, and the day-by-day path of the winning strategy: in-sample events
in blue, out-of-sample in green, the stock alone dotted.

One ticker is an illustration, not evidence; the evidence is the scoreboard. A path that turns
jagged after a large move is the mark, not the trade: a deep in-the-money leg trades rarely and its
last print can be days stale.

In [ ]:
in_counts = events.ticker.value_counts()
cands = [t for t in oos_events.ticker.unique() if t in in_counts.index]
vol = panel[panel.bucket == BASELINE_BUCKET].groupby("ticker")["atm_volume"].median()
if not cands:    # no ticker has events in both windows (rare tag or short window): fall back to the busiest in-sample name
    print(f"No ticker has a '{EVENT_TAG}' event in both the in-sample and out-of-sample windows; "
          "showing the ticker with the most in-sample events instead (no out-of-sample path will be drawn).")
    cands = list(in_counts.index)
assert cands, f"No priced in-sample events for '{EVENT_TAG}'; nothing to illustrate."
EXAMPLE = sorted(cands, key=lambda t: (in_counts[t], vol.get(t, 0)), reverse=True)[0]
print(f"Example ticker: {EXAMPLE} — {in_counts[EXAMPLE]} in-sample CFO appointments and "
      f"{(oos_events.ticker == EXAMPLE).sum()} out-of-sample, median ATM volume {vol.get(EXAMPLE, 0):.0f} contracts on the pre-event session.")

ex_priced = [pe for pe in priced + oos_priced if pe.ticker == EXAMPLE and pe.bucket == BASELINE_BUCKET]
ex_rows = []
for pe in ex_priced:
    sample = "in-sample" if pe.event_date <= pd.Timestamp(STUDY_END) else "out-of-sample"
    m = pe.marks(pe.t_0)
    for name, key in [("ATM call", "C_K"), ("ATM put", "P_K"), (f"call {OTM_PCT:.0%} OTM", f"C_U{OTM_PCT}"), (f"put {OTM_PCT:.0%} OTM", f"P_L{OTM_PCT}")]:
        ex_rows.append({"sample": sample, "filing_date": pe.event_date.date(), "entry": pe.t_0.date(), "expiry": pe.expiry.date(),
                        "spot at entry": round(pe.synthetic_spot(pe.t_0, m), 2), "leg": name, "contract": pe.legs[key].ticker,
                        "strike": pe.legs[key].strike, "close at entry": m[key]})
with pd.option_context("display.max_colwidth", 30, "display.width", 200):
    display(pd.DataFrame(ex_rows).set_index(["sample", "filing_date", "entry", "expiry", "spot at entry", "leg"]))

ex_res = pd.concat([results, oos_results])
ex_res = ex_res[(ex_res.ticker == EXAMPLE) & (ex_res.bucket == BASELINE_BUCKET) & (ex_res.entry == ENTRY) & (ex_res.otm == OTM_PCT)]
ex_tbl = (ex_res[ex_res.horizon.isin([5, 21, 42, "exp"])]
          .assign(sample=lambda d: np.where(d.event_date <= pd.Timestamp(STUDY_END), "in-sample", "out-of-sample"),
                  filing=lambda d: d.event_date.dt.date)
          .set_index(["sample", "filing", "horizon"])[["realized"] + STRATEGIES[1:]]
          .rename(columns=STRATEGY_LABEL).map(lambda v: f"{v * 100:+.1f}%" if pd.notna(v) else ""))
display(ex_tbl)


def pnl_path(pe: PricedEvent, strategy: str, entry: str = ENTRY, otm: float = OTM_PCT) -> pd.Series:
    """The strategy's P&L per $1 spot on every session from entry to expiry."""
    e_day = pe.t_pre if entry == "pre" else pe.t_0
    m_e = pe.marks(e_day)
    S_e = pe.synthetic_spot(e_day, m_e)
    days = CAL[(CAL >= e_day) & (CAL <= min(pe.expiry_session, LAST_SESSION))]
    out = {}
    for d in days:
        m = pe.marks(d)
        out[sessions_between(e_day, d)] = strategy_pnl(m_e, m, S_e, pe.synthetic_spot(d, m), otm)[strategy]
    return pd.Series(out, name=f"{pe.event_date.date()}")


fig, ax = plt.subplots(figsize=(10, 4.8))
for pe in ex_priced:
    sample = "in-sample" if pe.event_date <= pd.Timestamp(STUDY_END) else "oos"
    path = pnl_path(pe, WINNER)
    ax.plot(path.index, path.to_numpy() * 100, color=SERIES[sample], linewidth=2 if sample == "oos" else 1.4,
            alpha=1 if sample == "oos" else 0.85, label=f"{pe.event_date.date()} ({'out-of-sample' if sample == 'oos' else 'in-sample'}, exp {pe.expiry.date()})")
    ax.plot(path.index, pnl_path(pe, "stock").to_numpy() * 100, color=SERIES[sample], linewidth=0.8, linestyle=":", alpha=0.6)
ax.axhline(0, color=MUTED, linewidth=0.8)
style(ax, f"{EXAMPLE}: {STRATEGY_LABEL[WINNER]} after each CFO appointment · {BASELINE_BUCKET} options · entry = {ENTRY}",
      "sessions after entry", "P&L, % of spot at entry (dotted: the stock alone)")
ax.legend(frameon=False, labelcolor=INK2, fontsize=8)
plt.tight_layout()
plt.show()

## 10 · Parameter sensitivity

A finding that lives at one parameter setting is not a finding. Two grids, both already in the long
table so they cost nothing: OTM distance, and expiry bucket crossed with entry session. Read across
a row: does the sign hold, does the star survive, does the ranking change? Judges look for this
table, or the organizers' overfitting library run on the same grid.

In [ ]:
SENS_HORIZON = 21

grid_rows = []
for b in EXPIRY_BUCKETS:
    for e in ("pre", "post"):
        for o in OTM_GRID:
            sb = scoreboard(results, bucket=b, entry=e, otm=o, horizons=[SENS_HORIZON])
            for _, r in sb.iterrows():
                grid_rows.append({"bucket": b, "entry": e, "otm": o, "strategy": r.strategy, "n": r.n, "mean": r["mean"], "ci_lo": r.ci_lo, "ci_hi": r.ci_hi})
grid = pd.DataFrame(grid_rows)
grid["cell"] = [f"{m * 100:+.2f}%{'*' if (lo > 0 or hi < 0) else ''}" if pd.notna(m) else "" for m, lo, hi in zip(grid["mean"], grid.ci_lo, grid.ci_hi)]


def sens_table(sub: pd.DataFrame, columns: list[str]) -> pd.DataFrame:
    return (sub[sub.strategy != "stock"].pivot_table(index="strategy", columns=columns, values="cell", aggfunc="first")
              .reindex(STRATEGIES[1:]).rename(index=STRATEGY_LABEL).rename_axis(f"mean P&L at h={SENS_HORIZON}", axis=0))


print(f"1 · Vary the OTM distance of the sold call / bought put, holding the {BASELINE_BUCKET} bucket (in-sample, h={SENS_HORIZON}; * = CI excludes zero):")
display(sens_table(grid[grid.bucket == BASELINE_BUCKET], ["entry", "otm"]))
print(f"2 · Vary the expiry bucket and the entry session, holding OTM = {OTM_PCT:.0%}:")
display(sens_table(grid[grid.otm == OTM_PCT], ["entry", "bucket"]))

## 11 · Specifying the trade (scored on realism, not P&L)

If your finding is real, say how you would trade it and what it would cost. Judges want a
specification that survives contact with an options desk, not a Sharpe ratio. Cover at least:

- **Instrument and entry**: which of the five, which expiry bucket, filled at the close of `t_0`
  (tradeable if the filing was accepted before the bell, see the timing stretch) or the next open.
- **Exit**: a fixed horizon from the table, or expiry. Justify it from the decay of the edge across
  horizons, not from where it looks best.
- **Costs**: this notebook marks legs at their last trade, so it has no bid-ask spread. The cell
  below charges a flat fraction of each option premium per side as a stand-in. If your key includes
  options quotes (`/v3/quotes/O:…`), replace it with half the closing spread per leg and say what
  fraction of the gross edge it eats.
- **Capacity**: the `volume` column of each leg's bars is contracts traded that day. How much
  notional can your idea absorb before it moves the price?
- **Selection**: 8-Ks are filed *after* the event. Any `"pre"`-entry result is a statement about
  pricing, not a trade. Be explicit about which regime you are in.

The cell below is the minimum accounting for the in-sample winner at the 21-session horizon.

In [ ]:
COST_HAIRCUT = 0.05    # fraction of each option premium paid to get in and out (no quotes in this notebook, so an assumption)

hz = slice_results(results, BASELINE_BUCKET, ENTRY, OTM_PCT)
exp_rows = hz[hz.horizon == 21].copy()
legs_used = {"long_call": ["C_K"], "covered_call": [f"C_U{OTM_PCT}"], "protective_put": [f"P_L{OTM_PCT}"],
             "collar": [f"C_U{OTM_PCT}", f"P_L{OTM_PCT}"], "cash_secured_put": [f"P_L{OTM_PCT}"]}
prem = {(pe.ticker, pe.event_date): pe.marks(pe.t_0) for pe in priced if pe.bucket == BASELINE_BUCKET}
vols = {(pe.ticker, pe.event_date): sum(pe.legs[l].volume_on(pe.t_0) for l in legs_used[WINNER]) for pe in priced if pe.bucket == BASELINE_BUCKET}
exp_rows["premium_traded"] = [sum(abs(prem[(t, d)][l]) for l in legs_used[WINNER]) / s for t, d, s in zip(exp_rows.ticker, exp_rows.event_date, exp_rows.S_entry)]
exp_rows["cost"] = exp_rows["premium_traded"] * COST_HAIRCUT * 2
exp_rows["net"] = exp_rows[WINNER] - exp_rows["cost"]
exp_rows["leg_volume"] = [vols[(t, d)] for t, d in zip(exp_rows.ticker, exp_rows.event_date)]
summary = pd.Series({
    "strategy": STRATEGY_LABEL[WINNER],
    "events": f"{len(exp_rows)}",
    "gross mean P&L at h=21, per $1 spot": f"{exp_rows[WINNER].mean() * 100:+.2f}%",
    f"mean round-trip cost ({COST_HAIRCUT:.0%} of premium each way)": f"{exp_rows['cost'].mean() * 100:.2f}%",
    "net mean P&L at h=21": f"{exp_rows['net'].mean() * 100:+.2f}%",
    "share of events net positive": f"{(exp_rows['net'] > 0).mean():.0%}",
    "median contracts traded in the option legs on the filing session": f"{exp_rows['leg_volume'].median():,.0f}",
})
display(summary.to_frame("value"))

## Stretch · Timing: was the filing before or after the bell?

`filing_date` has no time of day, but every EDGAR submission header carries
`<ACCEPTANCE-DATETIME>`. A filing accepted at 17:30 could not be traded until the next session, so
its true filing session is the next day, and a `"post"` entry at the close of `filing_date` is
look-ahead. Flip `FETCH_ACCEPTANCE_TIMES` on, set a real `SEC_USER_AGENT` (the SEC blocks anonymous
clients and asks for at most 10 requests per second), and rerun from section 5. This pulls from
sec.gov, not from Massive, and is optional.

Going further: the 8-K is filed up to four business days *after* the event. Aligning to the press
release date named in `supporting_text` is the natural next refinement, and is exactly the kind of
judgment call the judges want to see argued.

In [ ]:
def fetch_acceptance_time(filing_url: str) -> pd.Timestamp | None:
    """Acceptance timestamp (US/Eastern) from the EDGAR submission header; None if not found."""
    key = CACHE_DIR / ("sec_" + hashlib.sha1(filing_url.encode()).hexdigest() + ".txt")
    if key.exists():
        head = key.read_text()
    else:
        resp = requests.get(filing_url, headers={"User-Agent": SEC_USER_AGENT}, timeout=30, stream=True)
        resp.raise_for_status()
        head = next(resp.iter_content(4096, decode_unicode=True))
        key.write_text(head)
        time.sleep(0.12)
    m = re.search(r"<ACCEPTANCE-DATETIME>(\d{14})", head)
    return pd.Timestamp(m.group(1)) if m else None


if FETCH_ACCEPTANCE_TIMES:
    events["accepted_at"] = [fetch_acceptance_time(u) for u in events["filing_url"]]
    after_close = events["accepted_at"].dt.hour >= 16
    events.loc[after_close, "t_0"] = [CAL[CAL.searchsorted(d, side="right")] for d in events.loc[after_close, "t_0"]]
    events["t_pre"] = events["t_0"].map(session_before)
    print(f"{after_close.mean():.0%} of filings were accepted after 16:00 ET; their filing session moved to the next day. "
          "Rerun from section 5 to re-price.")
else:
    print("Skipped. Set FETCH_ACCEPTANCE_TIMES = True and a real SEC_USER_AGENT to enable.")

## Sealed window · judges only

Everything above is a function of a tag, a start date and an end date. Judges set `RUN_HOLDOUT =
True` and the sealed dates in section 2, then run the notebook; this cell runs the whole pipeline on
that window and plots it against the in-sample result. Horizons that have not resolved by the
judging date are absent rather than guessed. A finding that replicates here is what wins; one that
vanishes is a finding too, and teams that predicted their own fragility in the write-up get credit
for it. Contestants will see "not run" here; that is expected.

In [ ]:
def run_study(tag: str, start: str, end: str, universe: list[str] = TOP_100, buckets: dict = EXPIRY_BUCKETS,
              max_events: int | None = None, label: str = None) -> dict:
    """The whole pipeline on a fresh window. Returns the events, priced legs, long results and the scoreboard."""
    ev = build_events(tag, start, end, universe)
    ev["event_date"] = ev["filing_date"]
    if max_events:
        ev = ev.head(max_events).copy()
    pr, drops = price_events(ev, buckets, label=label or f"{tag} {start}..{end}")
    res = evaluate(pr)
    return {"events": ev, "priced": pr, "dropped": drops, "results": res, "board": scoreboard(res)}


if RUN_HOLDOUT:
    holdout = run_study(EVENT_TAG, HOLDOUT_START, HOLDOUT_END)
    print(f"Sealed window {HOLDOUT_START}..{HOLDOUT_END}: n = {int(holdout['board'].n.max())}. "
          "Horizons that have not yet resolved are simply absent.")
    display(fmt_board(holdout["board"]))
    plot_scoreboards({"in-sample": board, "oos": holdout["board"]},
                     f"{EVENT_TAG}: in-sample vs sealed window {HOLDOUT_START}..{HOLDOUT_END} · {BASELINE_BUCKET} options")
else:
    print(f"Sealed window {HOLDOUT_START}..{HOLDOUT_END} not run. Judges: set RUN_HOLDOUT = True.")

## Appendix · Known limitations you should mention in your write-up

- **Static universe.** `TOP_100` is today's list applied to the whole window. Companies that grew
  into the top 100 during 2022–2026 are included for years when they were smaller, and companies that
  fell out are excluded for years when they were in. That is a survivorship bias; name it, and if you
  have a point-in-time membership source, use it.
- **The stock price is inferred, not observed.** Spot comes from put-call parity on the ATM pair
  with a flat carry rate (`RISK_FREE`) and no dividend term. Both legs are last trades, which can be
  minutes or hours stale. The error is typically a few tenths of a percent; the `abs_spot_gap`
  column in section 5 measures it for your run. It largely cancels in strategy P&L because entry and
  exit use the same construction.
- **The synthetic stock is not shares.** It collects no dividends, which matters over a 3–6 month
  hold on a high-yield name, and its short put can be assigned early in a way shares cannot. A covered
  call built on real shares would also face early assignment of the short call before an ex-dividend
  date. None of this is modelled.
- **Last-trade marks, no spreads.** Every price is the last trade of the day, so P&L is
  "theoretical": mid-market at best, and quiet contracts can carry a stale print for up to
  `MAX_STALE_SESSIONS` sessions. Deep in-the-money legs are the worst affected, which is why a
  covered call or collar path can look jagged after a large move. There is no bid-ask cost until you
  add quotes (section 11).
- **Small samples.** The worked example has about 35 events in-sample and 20 out-of-sample for one
  tag on 100 names. Intervals are wide by construction. Combining related tags (`cfo_departure`,
  `ceo_appointment`) or widening the universe are the two levers, each with its own cost.
- **No earnings flag.** Without an earnings calendar we cannot tell which events share their window
  with an earnings release, the single biggest confound. The 8-K itself is a partial fix: Item 2.02
  filings carry the earnings release, and the `financial_results` family of tags can identify some of
  them; coverage of those tags is thin, so treat it as a lower bound.
- **Filing lag and time of day.** `filing_date` is not the announcement date and has no time of day.
  See the timing stretch.
- **Strike availability.** When no strike sits at or beyond the requested OTM distance, the farthest
  listed one is used; check the `strikes` on any event that looks odd.
- **Coverage.** Filings & Disclosures begins January 2022 and is marked experimental; the taxonomy
  is versioned, so cite the version you used. Options history on contestant keys covers the
  in-sample and out-of-sample windows in section 2; do not move them earlier.